# Lecture 9: Duality

---

```{note}
In Lecture 8, we computed the shadow prices of the MTC Chennai bus-service LP as $\boldsymbol{\pi}^* = \mathbf{c}_B^\top \mathbf{B}^{-1} = [2, 1]$, and closed by promising that these numbers are themselves the optimal solution of a second, "mirror" linear program — the **dual**. This lecture makes good on that promise. We show how every LP (the **primal**) has an associated dual LP, how to construct it mechanically, how to solve it using the same BFS enumeration table from Lecture 7, and why its optimal value always equals the primal's optimal value (**strong duality**). Along the way, **complementary slackness** gives us a direct algebraic link between a primal BFS and its corresponding dual BFS — explaining, once and for all, where shadow prices come from.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Construct the dual of a linear program from its primal form using the standard max–min conversion rules.
2. Solve a dual LP using the BFS enumeration table (Lecture 7) and verify that its optimal value equals the primal optimal value (strong duality).
3. Apply complementary slackness to relate primal and dual optimal solutions, and interpret dual variables as shadow prices in transportation planning contexts.

**Prerequisites**: LP Formulation (Lecture 3), Graphical Method (Lecture 4), Python SciPy Method (Lecture 6), Simplex Method / BFS Table (Lecture 7), Sensitivity Analysis (Lecture 8)

**Estimated time**: 90 minutes (including in-class exercises)

---

## Why Duality?

Recall the MTC Chennai problem from Lecture 7. MTC runs $\text{E}_1$ (Premium Express) and $\text{E}_2$ (Limited-Stop Express) trips, constrained by 40 driver-shift-hours and 60 fuel units per day, earning a maximum contribution of $z^* = $ ₹140,000/day at $(x_1^*, x_2^*) = (20, 20)$.

Now imagine a private contractor approaches MTC with an offer: *"Lease us your driver-shift-hours and fuel allocation for the day, and we'll pay you a fixed price per unit of each resource."* MTC's depot manager must decide: **what is the minimum price per resource that makes leasing at least as attractive as running the buses themselves?**

If the contractor offers $y_1$ (₹'000) per driver-shift-hour and $y_2$ (₹'000) per fuel unit, MTC's running costs in resources used by each service must be *covered* by the lease payment, or MTC would rather keep running the service itself:

- Running one trip of $\text{E}_1$ uses 1 driver-shift-hour + 2 fuel units, earning ₹4,000. For leasing to be at least as good: $y_1(1) + y_2(2) \geq 4$.
- Running one trip of $\text{E}_2$ uses 1 driver-shift-hour + 1 fuel unit, earning ₹3,000. For leasing to be at least as good: $y_1(1) + y_2(1) \geq 3$.

The contractor, in turn, wants to minimize total lease payment $40y_1 + 60y_2$ subject to these constraints. This is precisely the **dual** of MTC's LP — a second optimization problem, derived mechanically from the first, whose optimal solution turns out to be exactly the shadow prices $\boldsymbol{\pi}^* = [2, 1]$ we computed in Lecture 8.

This is not a coincidence. **Every LP has a dual**, and the relationship between a problem (the **primal**) and its dual is one of the most powerful results in optimization theory — it underlies shadow-price economics, bounding arguments in algorithm design, and (later in this course) the dual simplex method.

---

## Formal Definition

**Definition**: Given a primal LP, the dual LP is constructed by (i) introducing one dual variable per primal constraint, (ii) swapping the roles of the objective coefficients $\mathbf{c}$ and the right-hand-side $\mathbf{b}$, (iii) transposing the constraint matrix $\mathbf{A}$, and (iv) flipping the optimization sense and constraint directions according to a fixed set of rules.

### Notation

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x} \in \mathbb{R}^n$ | Primal decision variables |
| $\mathbf{y} \in \mathbb{R}^m$ | Dual decision variables (one per primal constraint) |
| $z$ | Primal objective value |
| $w$ | Dual objective value |
| $\mathbf{c} \in \mathbb{R}^n$ | Primal objective coefficients $\to$ dual RHS |
| $\mathbf{b} \in \mathbb{R}^m$ | Primal RHS $\to$ dual objective coefficients |
| $\mathbf{A} \in \mathbb{R}^{m \times n}$ | Primal constraint matrix $\to$ dual uses $\mathbf{A}^\top$ |

### Conversion Rules

This course formulates LPs as maximization with $\leq$ constraints (Lecture 3). For this canonical form, the dual is:

$$
\begin{array}{c|c}
\textbf{Primal} & \textbf{Dual} \\
\hline
\max_{\mathbf{x}} \ z = \mathbf{c}^\top \mathbf{x} & \min_{\mathbf{y}} \ w = \mathbf{b}^\top \mathbf{y} \\
\mathbf{A}\mathbf{x} \leq \mathbf{b} & \mathbf{A}^\top \mathbf{y} \geq \mathbf{c} \\
\mathbf{x} \geq \mathbf{0} & \mathbf{y} \geq \mathbf{0}
\end{array}
$$

More generally, each primal constraint type and variable sign maps to a dual counterpart:

| Primal (max) | Dual (min) |
|---|---|
| $i$-th constraint is $\leq$ | $y_i \geq 0$ |
| $i$-th constraint is $\geq$ | $y_i \leq 0$ |
| $i$-th constraint is $=$ | $y_i$ free |
| $x_j \geq 0$ | $j$-th dual constraint is $\geq$ |
| $x_j$ free | $j$-th dual constraint is $=$ |

```{tip}
A useful memory device: **"max becomes min, $\mathbf{b}$ and $\mathbf{c}$ swap places, and $\mathbf{A}$ flips on its side ($\mathbf{A}^\top$)."** Every LP we have formulated so far in this course (Lectures 3, 7, 8) is max-with-$\leq$, so the first row of the table above is the one you will use throughout this lecture.
```

---

## In-Class Exercise

### Exercise 1

*Same as Lecture 7 In-Class Exercise 1*

The Chennai Metropolitan Transport Corporation (MTC) is evaluating two new express services on a feeder corridor:

| Service | Net contribution margin (₹'000/trip) |
|---------|----------------------------------------|
| $\text{E}_1$ — Premium Express | 4 |
| $\text{E}_2$ — Limited-Stop Express | 3 |

Two resources constrain how many trips of each service MTC can run per day:

- Driver-shift hours: both services require 1 driver-shift-hour per trip; 40 driver-shift-hours are available per day.
- Fuel allocation: $\text{E}_1$ (longer route) consumes 2 units of fuel per trip, $\text{E}_2$ consumes 1 unit; the daily fuel budget is 60 units.

Maximize net contribution.

**Primal**:
$$\max_{x_1,x_2} \ z = 4x_1 + 3x_2$$

$$\begin{aligned}
x_1 + x_2 &\leq 40 \quad \text{(driver-shift hours)} \\
2x_1 + x_2 &\leq 60 \quad \text{(fuel budget)} \\
x_1, x_2 &\geq 0
\end{aligned}$$

Applying the conversion rules: $\mathbf{c} = [4, 3] \to$ dual RHS; $\mathbf{b} = [40, 60] \to$ dual objective coefficients; $\mathbf{A} = \begin{bmatrix}1&1\\2&1\end{bmatrix} \to \mathbf{A}^\top = \begin{bmatrix}1&2\\1&1\end{bmatrix}$ — one row per primal **variable**, hence one dual **constraint** per primal variable.

**Dual**:
$$\min_{y_1,y_2} \ w = 40y_1 + 60y_2$$

$$\begin{aligned}
y_1 + 2y_2 &\geq 4 \quad \text{(dual constraint for } x_1\text{)} \\
y_1 + y_2 &\geq 3 \quad \text{(dual constraint for } x_2\text{)} \\
y_1, y_2 &\geq 0
\end{aligned}$$

Here, $y_1$ is the dual variable associated with the driver-shift-hour constraint, and $y_2$ is associated with the fuel-budget constraint — **one dual variable per primal resource**. This is exactly the "lease price" interpretation from the motivation above: $y_1$ is the price per driver-shift-hour, $y_2$ the price per fuel unit.

```{caution}
A common error is transposing $\mathbf{A}$ incorrectly. Always check dimensions: if the primal has $m$ constraints and $n$ variables, the dual must have $m$ variables and $n$ constraints. Here $m=n=2$, so this check alone won't catch a transposition error — write out $\mathbf{A}^\top$ explicitly and verify column-by-column.
```

---

### Solving the Dual via the BFS Table

The dual is itself an LP, so we solve it exactly as in Lecture 7: convert to standard form by subtracting **surplus variables** $e_1, e_2 \geq 0$ (since the constraints are $\geq$), then enumerate basic feasible solutions.

**Standard form**:
$$\begin{aligned}
y_1 + 2y_2 - e_1 &= 4 \\
y_1 + y_2 - e_2 &= 3 \\
y_1, y_2, e_1, e_2 &\geq 0
\end{aligned}$$

With $n+m = 2+2 = 4$ variables and $m=2$ equations, there are $\binom{4}{2}=6$ ways to choose the non-basic pair:

| Non-Basic (= 0) | Basic | $(y_1, y_2)$ | $(e_1, e_2)$ | Feasible? | $w = 40y_1+60y_2$ |
|------------------|-------|--------------|--------------|-----------|---------------------|
| $e_1, e_2$ | $y_1, y_2$ | $(2, 1)$ | $(0, 0)$ | Yes | $\mathbf{140}$ |
| $y_2, e_2$ | $y_1, e_1$ | $(3, 0)$ | $(-1, 0)$ | **No** | — |
| $y_2, e_1$ | $y_1, e_2$ | $(4, 0)$ | $(0, 1)$ | Yes | $160$ |
| $y_1, e_2$ | $y_2, e_1$ | $(0, 3)$ | $(2, 0)$ | Yes | $180$ |
| $y_1, e_1$ | $y_2, e_2$ | $(0, 2)$ | $(0, -1)$ | **No** | — |
| $y_1, y_2$ | $e_1, e_2$ | $(0, 0)$ | $(-4, -3)$ | **No** | — |

Since the dual is a minimization problem, the optimum is the smallest feasible $w$: $w^* = 140$ at $(y_1^*, y_2^*) = (2, 1)$.

```{tip}
Compare this table to Lecture 7's primal BFS table for the same problem — same mechanics (enumerate, check feasibility, evaluate objective), only the optimisation sense flips from "largest feasible $z$" to "smallest feasible $w$".
```

**This is exactly $\boldsymbol{\pi}^* = [2, 1]$ — the shadow prices computed in Lecture 8 directly from $\mathbf{c}_B^\top \mathbf{B}^{-1}$.** This is no coincidence: the optimal dual solution *is* the vector of shadow prices. Sensitivity analysis and duality are two views of the same underlying object.

---

### Strong Duality

Notice that the primal optimum and dual optimum are **equal**:

$$z^* = 140 = w^*$$

This is **strong duality**, one of the central theorems of linear programming:

```{note}
**Strong Duality Theorem**: If a primal LP has an optimal solution $\mathbf{x}^*$, then its dual also has an optimal solution $\mathbf{y}^*$, and $z^* = \mathbf{c}^\top\mathbf{x}^* = \mathbf{b}^\top\mathbf{y}^* = w^*$.
```

A weaker, more general fact holds even before optimality — **weak duality**: for *any* feasible $\mathbf{x}$ and *any* feasible $\mathbf{y}$, $\mathbf{c}^\top\mathbf{x} \leq \mathbf{b}^\top\mathbf{y}$. Every feasible dual solution gives an **upper bound** on the primal objective. Check this against the BFS table above: the feasible (but non-optimal) dual points give $w = 160$ and $w=180$, both $\geq z^* = 140$ — consistent with weak duality, since every feasible dual value upper-bounds every feasible primal value, and at optimality the bound is tight.

The gap $\mathbf{b}^\top\mathbf{y} - \mathbf{c}^\top\mathbf{x}$ for any feasible pair is called the **duality gap**; strong duality says this gap is exactly zero at the optimum for LPs (this is *not* generally true for non-linear or integer programs, where a gap may persist — a point we will return to in the NLP module).

---

### Complementary Slackness

Strong duality tells us $z^*=w^*$, but **complementary slackness** tells us *how* primal and dual optimal solutions are linked, variable by variable and constraint by constraint:

```{note}
**Complementary Slackness**: At the primal–dual optimum,
1. If a primal constraint is **not binding** (slack $s_i^* > 0$), the corresponding dual variable is zero: $y_i^* = 0$.
2. If a dual variable is **positive** ($y_i^* > 0$), the corresponding primal constraint **must be binding** ($s_i^* = 0$).
3. Symmetrically: if a primal variable is **positive** ($x_j^* > 0$), the corresponding dual constraint **must be binding** (surplus $e_j^* = 0$); if a dual constraint has slack (surplus $e_j^* > 0$), the corresponding primal variable is zero.
```

In compact form: $s_i^* \cdot y_i^* = 0$ for every $i$, and $e_j^* \cdot x_j^* = 0$ for every $j$ — at most one of each pair can be non-zero.

### Verifying Complementary Slackness for the MTC Example

| Primal | Value | Dual | Value | Product |
|---|---|---|---|---|
| $s_1^*$ (driver-shift slack) | $0$ | $y_1^*$ | $2$ | $0 \times 2 = 0$ ✓ |
| $s_2^*$ (fuel slack) | $0$ | $y_2^*$ | $1$ | $0 \times 1 = 0$ ✓ |
| $x_1^*$ (E$_1$ trips) | $20$ | $e_1^*$ (dual constraint-1 surplus) | $0$ | $20 \times 0 = 0$ ✓ |
| $x_2^*$ (E$_2$ trips) | $20$ | $e_2^*$ (dual constraint-2 surplus) | $0$ | $20 \times 0 = 0$ ✓ |

Both primal constraints are binding ($s_1^*=s_2^*=0$), so both dual variables are free to be positive — and indeed $y_1^*=2>0, y_2^*=1>0$. Both primal variables are positive ($x_1^*=x_2^*=20>0$), so both dual constraints must be binding — and indeed $e_1^*=e_2^*=0$. Every pair checks out.

```{tip}
Complementary slackness is the algebraic reason sensitivity analysis "worked for free" in Lecture 8: because the optimal primal basis corresponds exactly to the optimal dual basis, the same $\mathbf{B}^{-1}$ that solves the primal also generates the dual optimum $\boldsymbol{\pi}^* = \mathbf{c}_B^\top\mathbf{B}^{-1}$.
```

---

### Managerial Interpretation

Duality gives the MTC depot manager a second, equally valid way to read the same numbers:

> **Dual variable $y_1^*=2$**: The minimum price (₹2,000 per driver-shift-hour) at which MTC would be indifferent between running buses itself and leasing the hour to an outside contractor. Below this price, MTC keeps the hour; above it, MTC should lease it out. This is identical to the shadow price interpretation from Lecture 8 — duality explains *why* the shadow price is also a fair "resource price."

> **Dual variable $y_2^*=1$**: The minimum price (₹1,000 per fuel unit) at which fuel becomes worth selling rather than using. Any external offer above ₹1,000/unit makes leasing fuel more profitable than running $\text{E}_1$/$\text{E}_2$ trips.

> **Complementary slackness in practice**: Because both resources are fully used at the optimum ($s_1^*=s_2^*=0$), *both* carry a positive price — every unit of both resources is scarce and valuable. Had the fuel budget been generous enough that $s_2^* > 0$ at the optimum, complementary slackness tells us immediately that $y_2^*$ would have to be zero — unused resources are never worth paying for.

This dual reading — "the minimum price that makes leasing resources as good as using them" — is the standard economic interpretation of duality in resource-allocation problems, and it generalizes directly to the Transportation Problem (Lecture 10), where dual variables become **node potentials** (supply/demand prices) in network flow.

---

### Exercise 2

*Same as Lecture 7 Take-Away Exercise 2*

A Chennai-based logistics firm dispatches two container types from its yard each day: 20-ft containers ($x_1$) and 40-ft containers ($x_2$), earning net margins of ₹8,000 and ₹5,000 per dispatch (in ₹'000: $8$ and $5$).

Two yard resources constrain daily dispatches:

- Crane-hours: 4 hours/dispatch for 20-ft, 2 hours/dispatch for 40-ft; 60 crane-hours/day available.
- Gate-clearance hours: 2 hours/dispatch for 20-ft, 4 hours/dispatch for 40-ft; 48 gate-clearance-hours/day available.

Maximize net margings.

**Primal**:
$$\max_{x_1,x_2} \ z = 8x_1 + 5x_2$$

$$\begin{aligned}
4x_1 + 2x_2 &\leq 60 \quad \text{(crane-hours)} \\
2x_1 + 4x_2 &\leq 48 \quad \text{(gate-clearance hours)} \\
x_1, x_2 &\geq 0
\end{aligned}$$

**Task**: Construct the dual, solve it via the BFS table, and verify strong duality and complementary slackness against the known primal optimum $(x_1^*, x_2^*) = (12, 6)$, $z^* = 126$.

#### Step 1 — Construct the Dual

$\mathbf{c}=[8,5] \to$ dual RHS; $\mathbf{b}=[60,48] \to$ dual objective coefficients; $\mathbf{A}=\begin{bmatrix}4&2\\2&4\end{bmatrix} \to \mathbf{A}^\top = \begin{bmatrix}4&2\\2&4\end{bmatrix}$ (symmetric in this case).

$$\min_{y_1,y_2} \ w = 60y_1 + 48y_2$$

$$\begin{aligned}
4y_1 + 2y_2 &\geq 8 \quad \text{(dual constraint for } x_1\text{)} \\
2y_1 + 4y_2 &\geq 5 \quad \text{(dual constraint for } x_2\text{)} \\
y_1, y_2 &\geq 0
\end{aligned}$$

#### Step 2 — Solve via BFS Table

Standard form: $4y_1+2y_2-e_1=8$, $2y_1+4y_2-e_2=5$. Enumerating all $\binom{4}{2}=6$ bases:

| Non-Basic (= 0) | Basic | Basic values | Feasible? | $w = 60y_1+48y_2$ |
|------------------|-------|--------------|-----------|---------------------|
| $e_1, e_2$ | $y_1, y_2$ | $y_1=\frac{11}{6}, y_2=\frac{1}{3}$ | Yes | $\mathbf{126}$ |
| $y_2, e_2$ | $y_1, e_1$ | $y_1=\frac{5}{2}, e_1=2$ | Yes | $150$ |
| $y_2, e_1$ | $y_1, e_2$ | $y_1=2, e_2=-1$ | **No** | — |
| $y_1, e_2$ | $y_2, e_1$ | $y_2=\frac{5}{4}, e_1=-\frac{11}{2}$ | **No** | — |
| $y_1, e_1$ | $y_2, e_2$ | $y_2=4, e_2=11$ | Yes | $192$ |
| $y_1, y_2$ | $e_1, e_2$ | $e_1=-8, e_2=-5$ | **No** | — |

The minimum feasible $w$ is $\mathbf{w^* = 126}$ at $(y_1^*, y_2^*) = \left(\frac{11}{6}, \frac{1}{3}\right)$.

```{tip}
Non-integer dual solutions are common and perfectly valid — unlike primal trip/dispatch counts, dual variables are *prices*, which need not be whole numbers.
```

#### Step 3 — Verify Strong Duality

$$z^* = 126 = w^* \quad \checkmark$$

#### Step 4 — Verify Complementary Slackness

| Primal | Value | Dual | Value | Product |
|---|---|---|---|---|
| $s_1^*$ (crane-hour slack) | $0$ | $y_1^*$ | $11/6$ | $0$ ✓ |
| $s_2^*$ (gate-clearance slack) | $0$ | $y_2^*$ | $1/3$ | $0$ ✓ |
| $x_1^*$ | $12$ | $e_1^*$ | $0$ | $0$ ✓ |
| $x_2^*$ | $6$ | $e_2^*$ | $0$ | $0$ ✓ |

Both crane-hours and gate-clearance hours are fully utilised at the optimum, so both carry a strictly positive price — consistent with both $y_1^*, y_2^* > 0$.

---

## Take-Away Exercise

For the following problems, formulate the dual LP, solve using any method convinient to you (Excel Solve, Python SciPy, or Simplex by hand), and verify strong duality and complementary slackness for all constraint/variable pairs.

### Exercise 1

*Same as Lecture 7 Take-Away Exercise 1*

A Kochi-based freight operator must decide how many vehicle-trips of three types — Type A ($x_1$), Type B ($x_2$), Type C ($x_3$) — to deploy today, earning net profits of ₹50, ₹40, and ₹30 (in ₹'00/trip) respectively.

Three resources constrain the day's operations:

- Fuel: 3, 1, 2 units/trip for Types A, B, C; budget = 24 units.
- Loading-dock hours: 1, 2, 1 hours/trip; available = 16 hours.
- Driver shifts: 1, 1, 1 shift/trip; available = 10 shifts.

Maximize net profits.

$$\max_{x_1,x_2,x_3} \ z = 50x_1 + 40x_2 + 30x_3$$

$$\begin{aligned}
3x_1 + x_2 + 2x_3 &\leq 24 \quad \text{(fuel)} \\
x_1 + 2x_2 + x_3 &\leq 16 \quad \text{(loading-dock hours)} \\
x_1 + x_2 + x_3 &\leq 10 \quad \text{(driver shifts)} \\
x_1, x_2, x_3 &\geq 0
\end{aligned}$$

The primal optimum is $(x_1^*, x_2^*, x_3^*) = (7, 3, 0)$, $z^* = 470$.

### Exercise 2

*Same as Lecture 7 Take-Away Exercise 3*

An e-commerce company allocates its delivery fleet across three Bengaluru zones — Whitefield ($x_1$), Electronic City ($x_2$), and Koramangala ($x_3$). Net revenue per vehicle-day is ₹3,000, ₹5,000, and ₹4,000 (in ₹'000: $3, 5, 4$) respectively. Each zone's road network and parking can support only a limited numer of delivery vehicles (Whitefiled: 18, Electronic City: 12, Koramangala: 9) and the company has a limited fleet of 30 vehicles.

Maximize net revenue.

$$\max_{x_1,x_2,x_3} \ z = 3x_1 + 5x_2 + 4x_3$$

$$\begin{aligned}
x_1 &\leq 18 \quad \text{(Whitefield zone capacity)} \\
x_2 &\leq 12 \quad \text{(Electronic City zone capacity)} \\
x_3 &\leq 9 \quad \text{(Koramangala zone capacity)} \\
x_1 + x_2 + x_3 &\leq 30 \quad \text{(total fleet available)} \\
x_1, x_2, x_3 &\geq 0
\end{aligned}$$

The primal optimum is $(x_1^*, x_2^*, x_3^*) = (9, 12, 9)$, $z^* = 123$ — note that the Whitefield constraint is not binding ($x_1^*=9 < 18$).

---

## Further Reading

- Vanderbei, R.J. (2014). *Linear Programming: Foundations and Extensions* — Chapter 4: Duality Theory.
- Bertsimas, D. and Tsitsiklis, J.N. (1997). *Introduction to Linear Optimization* — Chapter 4: Duality Theory.
- Winston, W.L. (2004). *Operations Research: Applications and Algorithms* — Chapter 6: Duality and Sensitivity Analysis.
- SciPy documentation: `scipy.optimize.linprog` (`method='highs'`) — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)